# Milestone 4 — Transformer training (Experiments C and D)

**Run this top to bottom on a Colab T4.** Full instructions, secrets and runtime
estimate: `research/scripts/MILESTONE_4_COLAB_HANDOFF.md`.

Trains **mBERT (C)** and **XLM-R-base (D)** in-domain on Ax-to-Grind, 3 seeds each
(`{42, 123, 2026}`), writes metrics through the project's existing metrics contract,
and pushes each checkpoint to a **staging** HF repo.

Before you start:
1. **Runtime → Change runtime type → T4 GPU**
2. Add your HF **write** token to Colab **Secrets** (🔑 sidebar) as `HF_TOKEN`, with notebook access on
3. Set `HF_STAGING_PREFIX` in the config cell below to your HF username

## 1. Check the GPU

Stops immediately if there is no GPU, rather than silently spending hours on CPU.

In [ ]:
import subprocess, sys

out = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if out.returncode != 0:
    sys.exit("No GPU detected. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print(out.stdout)

## 2. Configuration

`HF_STAGING_PREFIX` is not secret — set it here. `HF_TOKEN` comes from Colab Secrets.

In [ ]:
# ---- EDIT THIS ----
HF_STAGING_PREFIX = "your-hf-username"   # e.g. "rehman-ayoub"
REPO_URL = "https://github.com/<your-github-username>/<repo-name>.git"
REPO_BRANCH = "main"
# -------------------

assert HF_STAGING_PREFIX != "your-hf-username", "Set HF_STAGING_PREFIX to your HF username first."
print("staging namespace:", HF_STAGING_PREFIX)

In [ ]:
import os

# Read the write token from Colab Secrets so it never appears in saved notebook output.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as exc:
    raise SystemExit(
        "Could not read HF_TOKEN from Colab Secrets. Open the key icon in the left "
        f"sidebar, add HF_TOKEN (role: write), enable notebook access, rerun. ({exc})"
    )

os.environ["HF_STAGING_PREFIX"] = HF_STAGING_PREFIX
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))
print("HF_STAGING_PREFIX set:", os.environ["HF_STAGING_PREFIX"])

## 3. Clone the repo and install pinned dependencies

In [ ]:
import os, subprocess

if not os.path.isdir("/content/repo"):
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, "/content/repo"], check=True)
os.chdir("/content/repo")
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Step 1 of 2 — remove Colab's preinstalled torchvision/torchaudio BEFORE installing.
#
# They are compiled against whatever torch Colab shipped with. The install below
# moves torch to this project's pin, and the leftovers then register C++ ops against
# the wrong ABI, so `from transformers import Trainer` dies with
#     RuntimeError: operator torchvision::nms does not exist
# transformers guards that import with is_torchvision_available(), which only checks
# whether the package is INSTALLED, not whether it imports — a present-but-broken
# copy passes the guard and raises RuntimeError, which nothing on that path catches.
# With torchvision absent the guard is simply False and the block is skipped.
# This project does zero vision and zero audio work. See research/requirements.txt.
#
# pip will warn that fastai/timm now have an unsatisfied torchvision requirement.
# That is expected and harmless — nothing in this pipeline imports them.
!pip uninstall -y -q torchvision torchaudio

# Step 2 of 2 — install the pinned set (REPRODUCIBILITY.md Section 1). On Linux/Colab
# the plain torch pin resolves to the CUDA build, which is what a T4 needs.
!pip install -q -r research/requirements.txt

In [ ]:
# Environment gate. Fails here, in two seconds, rather than an hour into training.
import importlib.util

import torch

assert importlib.util.find_spec("torchvision") is None, (
    "torchvision is still installed. Re-run the cell above; if it persists, "
    "Runtime -> Restart session and run this notebook from the top."
)

# The exact import that failed on the first Colab attempt. Kept as an assertion so a
# future environment drift is caught by this cell and not by the smoke test.
from transformers import Trainer  # noqa: F401
import transformers

print("torch       ", torch.__version__, "| cuda:", torch.cuda.is_available())
print("transformers", transformers.__version__, "| Trainer import OK")
print("python      ", __import__("sys").version.split()[0])
assert torch.cuda.is_available(), "torch cannot see the GPU — check the runtime type."

## 4. Data check

Training reads the **committed split index files**, so the raw corpora must be
present. `research/data/raw/` is gitignored, so this re-downloads it (Ax-to-Grind
needs no credentials; Notri-Fact is not used by this milestone).

In [ ]:
!python -m research.src.data.download --only ax_to_grind
!python -m research.src.data.validate

## 5. Smoke test — a few real steps before the real job

**If this fails, stop and send the error.** It costs a minute and catches setup
problems before an hour of GPU time is spent on them.

In [ ]:
!python -m research.src.models.transformer --dry-run --experiments D --dry-run-max-length 128

## 6. The real runs — 2 models × 3 seeds

~40–80 min total. Checkpoints push to staging as each seed finishes, so an
interrupted session does not lose completed work.

If you would rather split across two sessions, run the two cells separately.

In [ ]:
!python -m research.src.experiments.run_in_domain --models transformer --experiments C

In [ ]:
!python -m research.src.experiments.run_in_domain --models transformer --experiments D

## 7. Summary — copy this table back

In [ ]:
import glob, json

rows = []
for path in sorted(glob.glob("research/results/metrics/[CD]_*.json")):
    d = json.load(open(path, encoding="utf-8"))
    rows.append((
        d["experiment_id"], d["model"], d["split"], d["seed"],
        round(d["metrics"]["macro_f1"], 4),
        round(d["metrics"]["accuracy"], 4),
        d["prediction_collapse"]["is_collapsed"],
        d["run_metadata"].get("truncation", {}).get("pct_truncated"),
        round(d["run_metadata"].get("train_runtime_seconds", 0) / 60, 1),
    ))

hdr = f"{'exp':<4}{'model':<34}{'split':<6}{'seed':<7}{'macroF1':>9}{'acc':>8}{'collapsed':>11}{'trunc%':>8}{'min':>7}"
print(hdr); print("-" * len(hdr))
for r in rows:
    print(f"{r[0]:<4}{r[1]:<34}{r[2]:<6}{r[3]:<7}{r[4]:>9}{r[5]:>8}{str(r[6]):>11}{str(r[7]):>8}{r[8]:>7}")

print("\nSeed variance (test macro-F1) — report this, EXPERIMENT_PLAN.md Section 5:")
import statistics
for exp in ("C", "D"):
    vals = [r[4] for r in rows if r[0] == exp and r[2] == "test"]
    if len(vals) > 1:
        print(f"  {exp}: mean={statistics.mean(vals):.4f} sd={statistics.stdev(vals):.4f} values={vals}")

## 8. Package the metrics for download

Bring the zip back, plus the staging repo IDs and **revision SHAs** printed above
(`REPRODUCIBILITY.md` Section 6 needs repo ID *and* revision — a branch name moves,
a SHA does not).

In [ ]:
!cd research/results/metrics && zip -q /content/milestone4_metrics.zip [CD]_*.json && echo zipped

try:
    from google.colab import files
    files.download("/content/milestone4_metrics.zip")
except Exception as exc:
    print("Download the file manually from the Files pane:", exc)